# AtomicNG — TPU calibration (Colab)

Ночная калибровка всех бэкендов на TPU: probe → pytest → XLA BPTT-стенд → HF sidecar → обучение writer'а с бюджетом.
Метод: [arXiv:2610.31125](https://arxiv.org/abs/2610.31125). Repo: [Akseleu-J/atomicng](https://github.com/Akseleu-J/atomicng).

**Перед запуском:** Runtime → Change runtime type → **TPU**. Затем Runtime → Run all (Ctrl+F9).

In [ ]:
!pip install -q pytest safetensors transformers
!pip install -q git+https://github.com/Akseleu-J/atomicng.git
!git clone -q https://github.com/Akseleu-J/atomicng.git /content/atomicng-src
import atomicng, time
print("atomicng", atomicng.__version__)

In [ ]:
import json, multiprocessing
from pathlib import Path
import torch

RES = Path("/content/results"); RES.mkdir(exist_ok=True)
info = {"python": __import__("sys").version.split()[0], "torch": torch.__version__,
        "nproc": multiprocessing.cpu_count(), "cuda": torch.cuda.is_available()}
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    info["torch_xla"] = torch_xla.__version__
    info["xla_device"] = str(xm.xla_device())
    info["xla_devices"] = str(xm.get_xla_supported_devices())
except Exception as e:
    info["torch_xla"] = f"unavailable: {e!r}"
    print("torch_xla недоступен. Для Colab TPU установи:")
    print("  pip install 'torch_xla[tpu]' -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm.html")
    print("  (версию согласуй с предустановленным torch: см. https://cloud.google.com/tpu/pytorch-xla)")
try:
    import jax
    info["jax"] = f"{jax.__version__} {jax.devices()}"
except Exception as e:
    info["jax"] = f"unavailable: {e!r}"
(RES / "S0_probe.json").write_text(json.dumps(info, indent=2, default=str))
print(json.dumps(info, indent=2, default=str))

In [ ]:
# Полный набор тестов (сеть доступна: HF-тесты тоже отработают)
!cd /content/atomicng-src && python -m pytest tests -q -p no:cacheprovider

In [ ]:
# torch_xla обязан совпадать с torch по ABI (иначе undefined symbol в _XLAC.so).
# После установки: Runtime → Перезапустить сеанс → выполни S2b-continue.
import torch
torch_ver = torch.__version__.split('+')[0]  # например 2.14.1
print('предустановлен torch', torch.__version__)
!pip install -q "torch_xla[tpu]~={torch_ver}" -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm.html
print('ГОТОВО. Теперь: Среда выполнения → Перезапустить сеанс, затем выполни ячейку ниже (S2b-continue).')

In [ ]:
# Проверка после перезапуска: torch_xla жив?
try:
    import torch, torch_xla
    import torch_xla.core.xla_model as xm
    print('torch', torch.__version__, '| torch_xla', torch_xla.__version__, '|', xm.xla_device())
    print('XLA_OK — выполняй ячейку S3_accel_xla')
except Exception as e:
    print('XLA всё ещё не поднимается:', repr(e))
    print('Верная пара для Colab TPU (v5e): torch 2.14.x + torch_xla[tpu]~=2.14.0 (tpuvm index)')

In [ ]:
import json
from pathlib import Path
report = {"status": "SKIPPED_XLA_BROKEN",
          "fix": "перезапусти runtime после установки пары torch+torch_xla (см. ячейку установки)"}
Path('/content/results/S3_accel_xla.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

In [ ]:
# ═══ S3b_accel_jax: математика D1 на TPU-чипе через JAX (torch_xla не требуется) ═══
# Упрощённый стенд: read T⁰+Δ → writer-цели → D1-write, 20 итераций на чипе.
# BPTT-обучение калибруется XLA-путём после restart'а пары torch+torch_xla.
import time, json
import numpy as np
import jax, jax.numpy as jnp
from pathlib import Path

d, N, V = 32, 4096, 256
k1, k2, k3, k4 = jax.random.split(jax.random.PRNGKey(17), 4)
params = {
    'embed': 0.10 * jax.random.normal(k1, (V, d)),
    't0':    0.02 * jax.random.normal(k2, (N, d)),
    'w_out': 0.05 * jax.random.normal(k3, (d, V)),
    'w':     0.05 * jax.random.normal(k4, (d, d)),  # writer-цель = tanh(h @ w)
}
fact, noise = [5, 9, 13, 21, 33, 41], [2, 7, 11, 19, 23, 29, 31, 37, 43, 47]
block = np.array(fact + fact + noise)
T = len(block)
# CPU-хэш биграмм (grad-stop интерфейс — как в ядре)
big = [((block[i-1] * 2654435761 + block[i]) % N) for i in range(1, T)]
rows_idx = np.array([0] + big)

def d1_block(params, delta):
    ids = jnp.asarray(block)
    read = params['t0'][jnp.asarray(rows_idx[1:])] + delta[jnp.asarray(rows_idx[1:])]
    h = params['embed'][ids] + jnp.concatenate([jnp.zeros((1, d)), read])
    logits = h[:-1] @ params['w_out']
    ce = -jax.nn.log_softmax(logits)[jnp.arange(T - 1), ids[1:]].mean()
    targets = jnp.tanh(h[1:] @ params['w'])
    gates = jax.nn.sigmoid(jnp.full((targets.shape[0],), 0.5))  # скаляр на событие (T-1,)
    return ce, targets, gates

@jax.jit
def step_fn(params, delta):
    ce, targets, gates = d1_block(params, delta)
    a = (gates[:, None] * targets).mean(axis=0)
    w = gates.mean()
    rows_j = jnp.asarray(rows_idx[1:])
    delta = delta.at[rows_j].set((1 - w) * delta[rows_j] + a[None, :])  # D1: (1−w)Δ + A
    return params, delta, ce

delta = jnp.zeros((N, d))
losses, times = [], []
for step in range(20):
    t0 = time.perf_counter()
    params, delta, ce = step_fn(params, delta)
    jax.block_until_ready(ce)
    losses.append(round(float(ce), 4)); times.append(time.perf_counter() - t0)

report = {'steps': 20, 'device': str(jax.devices()[0]),
          'loss_first5': losses[:5], 'loss_last5': losses[-5:],
          'iter_time_s_mean': round(sum(times)/len(times), 4),
          'iter_time_s_max': round(max(times), 4),
          'note': 'D1 forward+write на чипе; BPTT-обучение — XLA-путь после restart пары'}
Path('/content/results').mkdir(exist_ok=True)
Path('/content/results/S3b_accel_jax.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

In [ ]:
# HF sidecar на настоящей модели: Δ_empty / retention
from transformers import AutoModelForCausalLM, AutoTokenizer
import atomicng
from atomicng.backends.hf import attach_sidecar
from atomicng.writers.mlp import WriterMLP

model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M").eval()
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
spec = atomicng.TableSpec(name="colab-hf-sidecar", n_rows=4096, row_width=16)
handle = attach_sidecar(model, spec=spec, inject_layer=2, seed=17)
writer = WriterMLP(hidden_size=int(model.config.hidden_size), row_width=16, hidden=64, target_scale=0.3)
docs = [f"Colab calibration document {i} about atomic rows with facts {i}." for i in range(10)]
deltas = []
for doc in docs[:8]:
    ids = tok.encode(doc, add_special_tokens=False)[:64]
    handle.memory.clear()
    empty = handle.nll([ids])
    handle.write(ids, writer)
    deltas.append(round(empty - handle.nll([ids]), 4))
handle.memory.clear(); handle.detach()
report = {"delta_empty_per_doc": deltas, "delta_empty_mean": round(sum(deltas)/len(deltas), 4)}
(RES / "S4_hf_sidecar.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

In [ ]:
# Обучение writer'а на Gutenberg-корпусе с бюджетом 45 минут + acquisition-eval
import urllib.request, random
from atomicng.backends.hf import SidecarTable, attach_sidecar
from atomicng.evidence import build_blocks
from atomicng.writers.hf_training import Block, HFWriterTrainer
from atomicng.writers.io import save_writer

corpus = Path("/content/corpus.txt")
if not corpus.exists():
    urllib.request.urlretrieve("https://www.gutenberg.org/cache/epub/1661/pg1661.txt", corpus)
docs = [p.strip() for p in corpus.read_text(encoding="utf-8-sig").split("\n\n") if len(p.strip()) >= 300]
rng = random.Random(17); rng.shuffle(docs)
eval_docs, train_docs = docs[:8], docs[8:]

from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M").eval()
spec = atomicng.TableSpec(name="colab-hf-train", n_rows=4096, row_width=16)
table = SidecarTable(spec, int(model.config.hidden_size), seed=17)
writer = WriterMLP(hidden_size=int(model.config.hidden_size), row_width=16, hidden=64, target_scale=0.3)
trainer = HFWriterTrainer(model, writer, spec, table, lr=3e-4, clip=1.0)

def doc_blocks(doc):
    ids = tok.encode(doc, add_special_tokens=False)
    return [Block(b.tokens, len(b.query)) for b in build_blocks(ids, block_size=64, query_prefix=16) if len(b.tokens) > 17]

all_blocks = [b for d in train_docs for b in doc_blocks(d)]
trajectories = [all_blocks[i:i+3] for i in range(0, len(all_blocks) - 2, 3)]
BUDGET_MIN = 45
losses, step, t0 = [], 0, time.monotonic()
while (time.monotonic() - t0) / 60 < BUDGET_MIN and step < 20000:
    loss = trainer.train_group([trajectories[step % len(trajectories)]])
    losses.append(round(loss, 4)); step += 1
    if step % 50 == 0:
        print(f"step {step}: loss {loss:.4f} | осталось {BUDGET_MIN - (time.monotonic()-t0)/60:.0f} мин", flush=True)
trainer.close(); writer.eval()
save_writer(writer, "/content/writer_colab.safetensors", extra={"steps": step, "session": "colab-tpu"})

handle = attach_sidecar(model, spec=spec, inject_layer=trainer.inject_layer, seed=17)
handle.table = table
deltas = []
for doc in eval_docs:
    ids = tok.encode(doc, add_special_tokens=False)[:64]
    if len(ids) < 8: continue
    handle.memory.clear()
    empty = handle.nll([ids]); handle.write(ids, writer)
    deltas.append(round(empty - handle.nll([ids]), 4))
handle.memory.clear(); handle.detach()
report = {"steps": step, "train_minutes": round((time.monotonic()-t0)/60, 1),
          "loss_first10": losses[:10], "loss_last10": losses[-10:], "loss_min": min(losses),
          "acquisition_delta_empty": deltas}
(RES / "S5_hf_train.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## Итог
Отчёты стадии — в `/content/results/*.json` (S0_probe, S3_accel_xla, S4_hf_sidecar, S5_hf_train), чекпоинт — `/content/writer_colab.safetensors`. Скачай их и положи в `kaggle-evidence/colab-tpu/` репо-папки evidence.